# Part 3: Towards Experimental Design for a Precessing Qubit
#### *Tutorial by [Alexandra Ramôa](https://sites.google.com/view/alexandraramoa)*

## Installs and imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize as opt
import pickle
from scipy.special import logsumexp
from copy import deepcopy

## 3.1 Unfolding Bayes' rule

Recall Bayes' rule as applied to the precession frequency example:
$$
P(\omega \mid D) = \frac{L(\omega \mid D)P(\omega)}{P(D)}
$$

Note that $D$ can be a dataset of any length. For instance, it can be a single datum, i.e. is a time-outcome tuple $(t, y)$. In this case, the likelihood is $L = \cos (\omega t/2)^{2y}\sin (\omega t/2)^{2(1-y)}$. If $D$ is a list of data, the likelihood is the product of the likelihoods of each one.

Alternatively, we can **sequentially update** the Bayesian distribution; the **posterior for one iteration becomes the prior for the next**. For instance, let's say we have 2 data, $D_0$ and $D_1$. We can either consider both at once:

$$
P(\omega \mid D_0,D_1) = \frac{L(\omega \mid D_0,D_1)P(\omega)}{P(D_0,D_1)}
$$

Or we can do two sequential updates:

$$
P(\omega \mid D_0) = \frac{L(\omega \mid D_0)P(\omega)}{P(D_0)}
$$

$$
P(\omega \mid D_0, D_1) = \frac{L(\omega \mid D_1)P(\omega \mid D_0)}{P(D_1)}
$$

It is easy to see that these 2 approaches (batch updating and sequential updating) are mathematically equivalent, and this generalizes to any number of data.

Sequential updating can be useful for applications such as real-time sensing, where we want to continuously estimate parameters based on cumulative datasets.  


## 3.2 Class implementation of Bayesian inference

Below is a dataset generation function for the precession phenomena from the previous notebook, and a class implementation of Bayesian inference. This class defines a uniform distribution at initialization, and provides methods for calculating and plotting likelihoods (for the precession example), performing Bayesian updates and calculating expectations.

Note that the attribute `ps` is a logarithm of the probability. All the calculations are done in logarithms for stability (avoiding undeflows due to multiplication of many small numbers), so products become sums.  

In [ ]:
def generate_dataset(w):
  ts = np.linspace(0, 10, 20)
  ps = np.cos(w*ts/2)**2
  ys = np.random.binomial(1, ps)
  return list(zip(ts, ys))

def split_data(data):
    ts = np.array([datum[0] for datum in data])
    ys = np.array([datum[1] for datum in data])
    return ts, ys

In [ ]:
class BayesianDistribution():
    def __init__(self, domain, Npoints = 100):
        self.ws =  np.linspace(domain[0], domain[1], Npoints)# np.random.uniform(domain[0],domain[1], size = Npoints)
        self.ps = np.ones(Npoints)/Npoints
        print("> Instantiated uniform prior distribution.")

    def mean_and_std(self):
        ws, ps = self.ws, self.ps
        logZ = logsumexp(ps)
        ps = np.exp(ps - logZ)
        mu = np.sum(ps * ws)
        std = np.sqrt(np.sum(ps * (ws - mu)**2))
        return mu, std

    def expectation(self, function, description, after_function = lambda x: x,
                    silent = True):
        ws, ps = self.ws, self.ps
        logZ = logsumexp(ps)
        ps = np.exp(ps - logZ)
        e = np.sum(ps * function(ws))
        e = after_function(e)
        if not silent:
            print(f"> Calculated expectation, {description} = {e:.2f}.")
        return e

    def Bayes_update(self, data, silent = True):
        ts, ys = split_data(data)
        ws, ps = self.ws, self.ps
        self.ps += self.loglikelihoods(data)
        if not silent:
            print(f"> Updated distribution based on {len(data)} data.")

    def loglikelihoods(self, data, eps = 1e-10):
        ts, ys = split_data(data)
        ws = self.ws
        args = np.outer(ws, ts) / 2.0
        p1s = np.cos(args) ** 2
        p1s = np.clip(p1s, eps, 1 - eps)
        Ls = np.sum(ys*np.log(p1s) + (1 - ys)*np.log1p(-p1s), axis=1)
        return Ls

    def print_info(self):
        mu, std = self.mean_and_std()
        print(f"> Mean and std: {mu:.2f} ± {std:.2f}.")

    def plot_loglikelihood(self, data, vline = None):
        fig, ax = plt.subplots(figsize=(7, 5), constrained_layout=True)

        ax.set_title("Log-likelihood")
        ax.plot(self.ws, self.loglikelihoods(data), color="black")

        ax.minorticks_on()
        ax.grid(which="major", linestyle="-", alpha=0.6)
        ax.grid(which="minor", linestyle=":", alpha=0.4)

        ax.set_xlabel("Value of $p_1$")
        ax.set_ylabel("Loglikelihood")

        if vline:
            ax.axvline(vline, color="red", linestyle="--")

        plt.show()

In [ ]:
domain = (0,1)
w = 0.3
print(f"> Real frequency value: {w}.")
data = generate_dataset(w)

> Real frequency value: 0.3.


## 3.3 Batch vs. sequential updating in practice

**Using the `BayesianDistribution` class, calculate the posterior distribution given the dataset `data` using the 2 approaches described in the text above: batch updating, and sequential updating. Compare the resulting distribution parameters, using the `BayesianDistribution.print_info` method (which uses `BayesianDistribution.mean_and_std`).**

In [ ]:
# Write your code below for the batch updating approach.

### Solution:
d1 = BayesianDistribution(domain)
d1.Bayes_update(data, silent = False)
d1.print_info()

> Instantiated uniform prior distribution.
> Updated distribution based on 20 data.
> Mean and std: -1150.50 ± nan.


/tmp/ipykernel_28735/3850659831.py:10: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(np.sum(ps * (ws - mu)**2))


In [ ]:
# Write your code below for the sequential updating approach.

### Solution:
d2 = BayesianDistribution(domain)
for datum in data:
    d2.Bayes_update([datum], silent = False)
d2.print_info()

> Instantiated uniform prior distribution.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Updated distribution based on 1 data.
> Mean and std: -1150.50 ± nan.


/tmp/ipykernel_28735/3850659831.py:10: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(np.sum(ps * (ws - mu)**2))


## 3.4 Expectation values

We extract information from the Bayesian posterior distribution by integrating over it to obtain expected values of useful functions of the parameters:

$$
\mathbb E_{P(\omega \mid D)}[f(\omega)] = \int f(\omega)P(\omega \mid D)d\omega
$$

Here we consider the particular case of expectations over a posterior distribution, but $P(\omega \mid D)$ can be replaced by any other distribution over $\omega$.

The distribution does not necessarily have a convenient closed form that allows analytical integration. Thus, we may use approximations such as numerical integration, where we discretized distribution to a set of points. In this case, integrals are replaced by sums:

$$
\mathbb E_{P(\omega \mid D)}[f(\omega)] \approx \sum f(\omega_i) \tilde P(\omega_i \mid D),
$$

where the tilde denotes a probability mass (which differs from the continuous probability evaluation due to normalization).

This is the approach done in the class above, and is already implemented in the method `BayesianDistribution.èxpectation`.

The mean and variance (or standard deviation) are particular cases of this.

- The mean is the expectation of the identity function, $f(\omega) = \omega$, and estimates the true parameter.

- The variance (or standard deviation) is the expectation of the squared deviation from the mean $f(\omega) = (\omega - \hat \omega)^2$ (or the square root of this expectation, taken *after* integration), and estimates the mean squared error (or root mean squared error).



**Take one of the posterior distributions you obtained in the previous sections. Recalculate the mean and standard deviation, but now using the `BayesianDistribution.expectation` method (instead of `BayesianDistribution.mean_and_std`). Compare the results.**

In [ ]:
# Write your code below.

### Solution:
d = d1
mean = d1.expectation(function = lambda x: x,
                      description = "mean",
                      silent = False)
std = d1.expectation(lambda x: (x - mean)**2,
                     description = "variance",
                     after_function = lambda x: x**0.5,
                     silent = False)

> Calculated expectation, mean = 0.31.
> Calculated expectation, variance = 0.05.



## 3.5 Utility forecasts

Let's say we want to perform an experiment to obtain a datum $(t, y)$ for a precessing qubit. The measurement time $t$ is the time elapsed between initialization and measurement; we can choose this time for each experiment, so we call $t$ an *experimental control*. Our goal is to obtain the smallest possible uncertainty (standard deviation of the Bayesian posterior distribution). **How should we choose $t$?**

For any prospective $t$, we can calculate the expected standard deviation over the prior Bayesian distribution. Let's first assume we want to choose between two possibilities, $t_a$ and $t_b$. For each of these candidate times:

- We can compute the standard deviation that would result from each possible scenario, i.e. outcome $y=0$ or $y=1$. This is done by performing a hypothetical Bayesian update on the datum, e.g. $(t_a, 0)$, and calculating the posterior standard deviation. These are the same operations we already did above.
- We can estimate the probability of each possible scenario, i.e. outcome $y=0$ or $y=1$. For example, we can estimate $P(y)$ for $t_a$ as the expectation of the probability $P(D \mid \omega)$ over the posterior distribution, where $D=(t_a,y)$. Note that we already know this probability: it is by definition the likelihood! Remember that $L(\omega \mid D) = P(D \mid \omega)$.

Thus

$$ \text{Expected std for } t_a =  P(0)*\left[\text{expected std for }D=(t_a,0)\right] + P(1)*[\text{expected std for }D=(t_a,1)]$$

That is, for any potential experimental control (here measurement time), we can **predict the standard deviation** that will result from the ensuing Bayesian update. We can then, for instance, choose among $t_a$ and $t_b$ the control that is expected to result in a smaller standard deviation.





**Create a function `conditional_std(d, t, y)` that given a current distribution `d`, a potential measurement time `t` and an outcome `y`, calculates the standard deviation that would result from Bayes-updating the distribution `d` after the datum $D=(t,y)$.**

In [ ]:
# Write your code below.

### Solution:
def conditional_std(d, t, y):
    cd = deepcopy(d)
    cd.Bayes_update([(t, y)])
    cstd = cd.mean_and_std()[1]
    # print(f"> Expected std for datum {(t, y)}: {cstd:.4f}")
    return cstd

conditional_std(d, 1, 0)

/tmp/ipykernel_28735/3850659831.py:10: RuntimeWarning: invalid value encountered in sqrt
  std = np.sqrt(np.sum(ps * (ws - mu)**2))


np.float64(nan)

**Create a function `expected_py(d, t, y)` that, given a distribution `d`, a time `t`  and an outcome `y`, calculates the expected probability of measuring `y`($0$ or $1$) upon measuring at time $t$.**

In [ ]:
# Write your code below.

### Solution:
def likelihood(w, t, y):
    if y==1:
        return np.cos(w*t/2)**2
    if y==0:
        return np.sin(w*t/2)**2

def expected_py(d, t, y):
    py = lambda w: likelihood(w, t, y)
    epy = d.expectation(py, description = f"p{y}")
    return epy

**Create a function `expected_std(d,t)` that, given a distribution `d` and a measurement time `t`, predicts the average standard deviation that would result from a measurement at time `t`.**

In [ ]:
# Write your code below.

### Solution:
def expected_std(d, t):
    ep1 =  expected_py(d, t, 1)
    ep0 =  expected_py(d, t, 0)

    std_0 = conditional_std(d, t, 0)
    std_1 = conditional_std(d, t, 1)

    exp_std = ep1 * std_1 + ep0 * std_0

    print(f"> Expected std for {t = }: {exp_std:.4f}.")

expected_std(d, 10)
expected_std(d, 0)

## 3.6 Bayesian experimental design

More generally, we can compute the *utility* of any experimental control, where the utility is a chosen function of the parameters. Here we choose the standard deviation, a standard choice, but we could want to minimize the interquartile range, etc.

Armed with this ability, we can optimize the experimental control. The simplest case would be evaluating utilities on a uniform distribution of candidate controls, but we can also employ more sophisticated optimization strategies.

What if we want to do $N$ experiments? Then we must optimize a vector of $N$ measurement times, $\vec t = \{t_1, \dots, t_N\}$. We can follow the same reasoning as above, the cost scales exponentially with $N$ (since there are $2^N$ possible scenarios for the $N$ binary outcomes), which quickly becomes intractable.

## 3.6 Putting the pieces together: adaptive approaches

Now comes the interesting part! The sequential nature of Bayesian updates and the ability to calculate expectations can combine into a useful power.

Alternatively to optimizing of a full vector of controls, we can optimize a single control at a time: optimize $t$, make the measurement, update the distribution depending on the outcome, make the measurement, and so on. This is called a locally optimal, or greedy, approach. Note that it is not necessarily globally optimal: the best choice for a single-experiment horizon may not be the best for e.g. a two-experiment horizon.

For instance, imagine that:

- Measuring at $t=0$ and $t=10$ results in a $5\%$ and then $5\%$ decrease in standard deviation.
- Measuring at $t=3$ and $t=8$ results in a $4\%$ and then $15\%$ decrease in standard deviation.

The latter case is better, yet the greedy approach would choose $t=0$ for the first measurement.

Despite this, the locally optimal approach tends to work well in practice, and has been widely adopted in quantum science. It also eliminates the exponential complexity in $N$; the cost becomes linear.   

Lastly, it has one significant benefit. Note that **the utility expectations are taken over the prior**; the predictions are only as accurate as the prior, which encodes pre-existing information. But this information may be very limited! We've been assuming all we know are the upper and lower bounds for $\omega$.

In the greedy case, this is remedied by the fact that as we collect information, we immediately use it to update our distribution, and it is based on this improved distribution that we optimize the subsequent controls. This stands in contrast with the global optimization case, where all measurement times are optimized in advance, and thus do not benefit from the incoming information.

In section 3.3, you found that batch updating and sequential updating yield the same results, *assuming the dataset is the same*. But importantly, sequential updating allows us to **adaptively choose the times**, and thus **obtain a better dataset**!
